# Lab 3: Neural Network Fundamentals - Linear Layers and Tensor Operations

## Lab Overview

This lab explores the fundamental building blocks of neural networks, focusing on linear transformations, tensor operations, and the mathematical foundations that power transformer architectures.

## Learning Objectives

By the end of this lab, you will:
- Understand linear layers and their role in neural networks
- Master tensor operations and broadcasting in PyTorch
- Learn about weight matrices and bias vectors
- Explore matrix multiplication in the context of neural networks
- Set up AMD GPU backend for efficient tensor computations
- Understand how linear transformations work in transformer models

---

## Step 1: Environment Setup

We'll configure our environment for neural network fundamentals using AMD GPU backend for optimal performance.

In [ ]:
# Initialize AMD GPU Backend and Required Libraries
import sys
sys.path.append('../')



# Core libraries for neural network fundamentals
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

AMD GPU environment initialized successfully
Using device: cuda
PyTorch version: 2.7.0
GPU: AMD Radeon Graphics
GPU Memory: 65.2 GB


## Step 2: Understanding Linear Layers

Linear layers (also called fully connected or dense layers) are fundamental components of neural networks. They perform the operation: **y = xW^T + b**

**Components:**
- **Weight Matrix (W)**: Learnable parameters that transform inputs
- **Bias Vector (b)**: Learnable offset added to the transformation
- **Input (x)**: Data to be transformed
- **Output (y)**: Transformed result

**In Transformers:**
- Query, Key, Value projections in attention
- Feed-forward network layers
- Output projection layers


In [2]:
# Create a linear layer: 2 input features -> 3 output features
linear = nn.Linear(in_features=2, out_features=3)

# Move to AMD GPU
linear = linear.to(device)

print("Linear Layer Analysis:")
print(f"Input features: {linear.in_features}")
print(f"Output features: {linear.out_features}")
print(f"Weight matrix shape: {linear.weight.shape}")  # [out_features, in_features]
print(f"Bias vector shape: {linear.bias.shape}")      # [out_features]
print(f"Device: {linear.weight.device}")

print("\nWeight Matrix:")
print(linear.weight.data)
print("\nBias Vector:")
print(linear.bias.data)

# Count parameters
total_params = sum(p.numel() for p in linear.parameters())
print(f"\nTotal parameters: {total_params}")
print(f"   Weights: {linear.weight.numel()}")
print(f"   Bias: {linear.bias.numel()}")

Linear Layer Analysis:
Input features: 2
Output features: 3
Weight matrix shape: torch.Size([3, 2])
Bias vector shape: torch.Size([3])
Device: cuda:0

Weight Matrix:
tensor([[ 0.5406,  0.5869],
        [-0.1657,  0.6496],
        [-0.1549,  0.1427]], device='cuda:0')

Bias Vector:
tensor([-0.3443,  0.4153,  0.6233], device='cuda:0')

Total parameters: 9
   Weights: 6
   Bias: 3


## Step 3: Tensor Operations and Matrix Multiplication

Now let's explore how linear layers process multi-dimensional tensors and understand the mathematical operations involved.

**Key Operations:**
1. **Matrix Multiplication**: Core operation in linear layers
2. **Tensor Broadcasting**: Automatic dimension expansion for operations
3. **Batch Processing**: Handling multiple samples simultaneously
4. **GPU Acceleration**: Parallel computation on AMD hardware

**Mathematical Details:**
- For input `x` with shape `[batch_size, ..., in_features]`
- Weight `W` with shape `[out_features, in_features]`
- Output `y = x @ W.T + b` with shape `[batch_size, ..., out_features]`

### **Exercise: Deconstructing the Linear Layer**

The `nn.Linear` layer is more than just a simple matrix multiplication. It's a powerful module that can operate on multi-dimensional tensors by applying the transformation to the last dimension. In this exercise, you will manually replicate its behavior to understand its inner workings.

**Your Task:**

1.  **Manual Matrix Multiplication**: Replicate the core `x @ W.T` operation using `torch.matmul`. Remember that the layer's `weight` attribute is already stored in a transposed form (`out_features`, `in_features`), so you will need to transpose it back for the multiplication.
2.  **Manual Bias Broadcasting**: Add the bias vector to the result of your matrix multiplication. PyTorch will automatically **broadcast** the bias, expanding its dimensions to match the shape of the matrix multiplication result.
3.  **`torch.einsum` Implementation**: Use Einstein summation (`torch.einsum`) to perform the matrix multiplication. This is an advanced and powerful way to express tensor operations. You'll need to define the correct string notation to map input dimensions to output dimensions.
4.  **Verification**: Verify that the output from the original `nn.Linear` layer, your manual implementation, and your `einsum` implementation are all identical.



In [ ]:
TODO=None

# Define a linear layer
in_features, out_features = 5, 3
linear = nn.Linear(in_features, out_features).to(device)

# Create a high-dimensional input tensor: [batch, sequence, height, in_features]
x = torch.randn(2, 4, 8, in_features, device=device)

# Get the official output from the nn.Linear layer
y_official = linear(x)

print("--- Setup ---")
print(f"Input Tensor Shape:  {x.shape}")
print(f"Linear Layer Weight Shape: {linear.weight.shape} (out_features, in_features)")
print(f"Linear Layer Bias Shape:   {linear.bias.shape} (out_features)")
print(f"Official Output Shape: {y_official.shape}")
print("-" * 30)

# --- Part 1: Manual Matmul and Bias Broadcasting ---
print("\n--- Implementing Manually (Matmul + Bias) ---")

# 1. Manually perform the matrix multiplication: y = x @ W.T
# Note: linear.weight has shape [out_features, in_features].
# You need to transpose it to [in_features, out_features] for the matmul.
y_matmul = TODO = None

# 2. Add the bias vector. PyTorch will broadcast the bias tensor
# of shape [3] to match the shape of y_matmul.
y_manual = TODO = None

print(f"Shape after manual matmul: {y_matmul.shape}")
print(f"Shape after adding bias:   {y_manual.shape}")

# --- Part 2: Advanced Implementation with torch.einsum ---
print("\n--- Implementing with Einstein Summation (einsum) ---")

# 3. Use torch.einsum to perform the matrix multiplication.
# The string should map the dimensions. A common notation for a batch matrix multiply is '...ij,...kj->...ik'.
# '...ij' represents our input tensor x (with last two dims i,j)
# '...kj' represents the weight tensor W (with dims k,j)
# '...ik' represents the desired output (with last two dims i,k)
# Remember that linear.weight is already W.T, so its shape is (k, j).
einsum_string = TODO = None
y_einsum_matmul = torch.einsum(einsum_string, x, linear.weight)

# Add the bias to the einsum result.
y_einsum = TODO = None

print(f"Einsum string used: '{einsum_string}'")
print(f"Shape after einsum matmul: {y_einsum_matmul.shape}")
print(f"Shape after adding bias:   {y_einsum.shape}")


# --- Part 3: Verification ---
print("\n--- Verification ---")

# 4. Use torch.allclose() to verify that your manual and einsum
# implementations match the official nn.Linear layer output.
manual_matches = TODO = None
einsum_matches = TODO = None

print(f"Manual implementation matches official output: {manual_matches}")
print(f"Einsum implementation matches official output: {einsum_matches}")

assert manual_matches and einsum_matches, "Verification failed! Your implementations are incorrect."
print("\n🎉 Verification successful! All methods produce identical results.")



## Step 4: Tensor Broadcasting

Broadcasting is a powerful feature that allows operations between tensors of different shapes. It's crucial for efficient neural network computations.

**Broadcasting Rules:**
1. Align shapes from the rightmost dimension
2. Dimensions of size 1 are stretched to match
3. Missing dimensions are treated as size 1
4. Incompatible shapes raise errors

**Applications in Neural Networks:**
- Adding bias vectors to matrix multiplication results
- Element-wise operations with different shaped tensors
- Scaling and normalization operations



### **Exercise: Per-Channel Normalization Using Broadcasting**

Broadcasting is a fundamental concept for writing efficient, vectorized code. In this exercise, you will use broadcasting to perform a common data science task: **per-channel normalization** of a batch of images.

Imagine you have a batch of images, represented by a 4D tensor of shape `[batch, channels, height, width]`. Your goal is to subtract the mean and divide by the standard deviation for **each channel independently**.

**Your Task:**

1.  **Calculate Statistics**: Compute the `mean` and `std` for each channel across the entire batch of images. The result should be two 1D tensors.
2.  **Reshape for Broadcasting**: The calculated `mean` and `std` tensors are 1D, but your image tensor is 4D. You must reshape the `mean` and `std` tensors so they can be broadcast correctly across the `batch`, `height`, and `width` dimensions.
3.  **Apply Normalization**: Use the reshaped tensors to normalize the input data: `normalized_data = (data - mean) / std`.
4.  **Challenge**: Apply a per-batch `row_mask` to the normalized data. The mask has a different shape and will test your understanding of broadcasting rules from another angle.
5.  **Debug an Error**: An example of an invalid broadcasting operation is provided. Your task is to reshape the tensor to make the operation valid.



In [4]:
# --- Setup ---
# A batch of 8 images, with 3 channels (RGB), and 64x64 resolution.
batch_size, channels, height, width = 8, 3, 64, 64
image_batch = torch.randn(batch_size, channels, height, width)

print("--- Data Normalization Exercise ---")
print(f"Input image batch shape: {image_batch.shape}")

# --- 1. Calculate Per-Channel Statistics ---
# Calculate the mean and std along the batch, height, and width dimensions (0, 2, 3).
# The result should be a tensor of shape [channels].
# Hint: Use the `dim` argument in torch.mean() and torch.std().
channel_mean = TODO = None
channel_std = TODO = None

print(f"\nCalculated channel mean shape: {channel_mean.shape}")
print(f"Calculated channel std shape:  {channel_std.shape}")
assert channel_mean.shape == (channels,)

# --- 2. Reshape Statistics for Broadcasting ---
# To subtract a [C] tensor from a [B, C, H, W] tensor, we need to add
# 'dummy' dimensions of size 1. Reshape the mean and std to [1, C, 1, 1].
mean_reshaped = TODO = None
std_reshaped = TODO = None

print(f"\nReshaped mean for broadcasting: {mean_reshaped.shape}")
print(f"Reshaped std for broadcasting:  {std_reshaped.shape}")
assert mean_reshaped.shape == (1, channels, 1, 1)

# --- 3. Apply Normalization ---
# Broadcasting will automatically expand the [1, C, 1, 1] tensors
# to match the [B, C, H, W] shape during the operations.
normalized_batch = TODO = None

print(f"\nShape after normalization: {normalized_batch.shape}")
assert normalized_batch.shape == image_batch.shape
print("Normalization successful!")

# --- 4. Challenge: Apply a Row Mask ---
# Create a mask that is different for each item in the batch but the same
# for every pixel in a given row. Shape: [B, 1, H, 1]
row_mask = torch.randn(batch_size, 1, height, 1)

# Multiply the normalized_batch by the row_mask.
# Broadcasting will expand the mask's channel and width dimensions.
masked_batch = TODO = None

print(f"\nChallenge: Mask shape: {row_mask.shape}")
print(f"Shape after applying mask: {masked_batch.shape}")
assert masked_batch.shape == image_batch.shape
print("Masking successful!")


# --- 5. Debugging: Fix an Invalid Broadcasting Operation ---
print("\n--- Debugging Challenge ---")
# This operation will fail because the trailing dimensions (4 vs 3) are not compatible.
# a.shape = (2, 4)
# b.shape = (3,)  -> Mismatched trailing dimension
a = torch.randn(2, 4)
b = torch.randn(3)
print(f"Shape of a: {a.shape}")
print(f"Shape of b: {b.shape}")
print("Operation `a + b` will fail.")

# Reshape tensor 'a' or 'b' to make them broadcast-compatible.
# For example, reshape 'b' to (3, 1) so it can broadcast against 'a' of shape (2, 4),
# resulting in an output of shape (2, 3, 4) if we add a new dim to a,
# or reshape 'a' to be compatible with 'b'.
# A simpler fix:

Broadcasting Examples:
Tensor a shape: torch.Size([2, 3, 4])
Scalar b: 10.0
a + b result shape: torch.Size([2, 3, 4])
First element: a[0,0,0] = 0.139, result[0,0,0] = 10.139

Tensor c shape: torch.Size([2, 3, 1])
Tensor d shape: torch.Size([4])
c + d result shape: torch.Size([2, 3, 4])

Broadcasting visualization:
c[0,0,0] = -0.519 gets added to each element of d
d = tensor([-0.6974, -1.8688, -0.8832, -1.6627], device='cuda:0')
Result c[0,0,:] + d = tensor([-1.2161, -2.3876, -1.4019, -2.1815], device='cuda:0')

Attention scores shape: torch.Size([2, 4, 8, 8])
Positional bias shape: torch.Size([1, 1, 8, 8])
Result shape: torch.Size([2, 4, 8, 8])

Broadcasting enables efficient operations without explicit tensor reshaping!

Manual expansion vs Broadcasting:
Manually expanded bias shape: torch.Size([2, 4, 8, 8])
Memory usage - Original bias: 64 elements
Memory usage - Expanded bias: 512 elements
Broadcasting saves memory by not creating copies!


## Step 5: Practical Applications in Transformers

Let's see how these linear operations are used in transformer architectures with practical examples.

**Transformer Applications:**
1. **Attention Projections**: Q, K, V matrices
2. **Feed-Forward Networks**: Two linear layers with activation
3. **Output Projection**: Final linear layer for vocabulary prediction
4. **Layer Normalization**: Learned scale and shift parameters


### **Exercise: Building a Complete Transformer Encoder Block**

You've seen how individual linear layers are used in Transformers. Now, you will combine these concepts to build a complete, functional **Transformer Encoder Block**. This block is the fundamental repeating unit of models like BERT and GPT.

A Transformer block consists of two main sub-modules:

1.  **Multi-Head Self-Attention (MHSA)**: Allows the model to weigh the importance of different tokens in the input sequence.
2.  **Position-wise Feed-Forward Network (FFN)**: A two-layer MLP applied independently to each token.

Both sub-modules are wrapped with **residual connections** and **Layer Normalization**.

**Your Task:**

1.  **Initialize Layers**: In the `__init__` method, define all the necessary layers: a combined projection for Q, K, V, an output projection for the attention module, the FFN layers, and two Layer Normalization modules.
2.  **Implement Multi-Head Logic**: In the `forward` pass, take the output of the combined QKV projection and reshape it to separate the attention heads.
3.  **Calculate Scaled Dot-Product Attention**: Implement the core attention mechanism: $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$.
4.  **Implement Residual Connections**: After the MHSA and FFN sub-modules, add the input of the module to its output (the residual connection) and then apply Layer Normalization.



In [ ]:
import math
class TransformerBlock(nn.Module):
    """A complete Transformer Encoder Block with Multi-Head Attention and FFN."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # --- 1. Initialize Layers ---
        # A single linear layer to project inputs to Q, K, and V combined.
        # This is an efficient implementation detail.
        self.qkv_proj = TODO = None

        # A final linear layer for the output of the attention mechanism.
        self.out_proj = TODO = None

        # The two-layer Feed-Forward Network (FFN).
        self.ffn = nn.Sequential(
            TODO = None, # Linear: embed_dim -> ff_dim
            TODO = None, # Activation (e.g., ReLU)
            TODO = None  # Linear: ff_dim -> embed_dim
        )

        # Layer Normalization modules
        self.norm1 = TODO = None
        self.norm2 = TODO = None

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_length, _ = x.shape

        # --- 2. Multi-Head Attention Logic ---
        # a. Project to QKV and reshape for multi-head processing.
        # qkv will have shape [B, S, 3 * D].
        qkv = self.qkv_proj(x)

        # Reshape qkv to [B, S, 3, H, D_h] and separate into Q, K, V.
        # H = num_heads, D_h = head_dim
        qkv = qkv.reshape(batch_size, seq_length, 3, self.num_heads, self.head_dim)
        # Permute to get shape [3, B, H, S, D_h] for easier splitting.
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = TODO = None, TODO = None, TODO = None # Split along the first dimension

        # --- 3. Scaled Dot-Product Attention ---
        # b. Calculate attention scores: (Q @ K.T) / sqrt(d_k)
        # K needs to be transposed on its last two dimensions.
        k_transposed = TODO = None
        scores = torch.matmul(q, k_transposed) / math.sqrt(self.head_dim)

        # c. Apply softmax to get attention weights, then apply to V.
        attention_weights = F.softmax(scores, dim=-1)
        attention_output = TODO = None # torch.matmul(attention_weights, v)

        # d. Concatenate heads and apply final output projection.
        # Reshape attention_output from [B, H, S, D_h] to [B, S, H * D_h = D]
        attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.embed_dim)
        attention_output = self.out_proj(attention_output)

        # --- 4. First Residual Connection and Layer Norm ---
        # x = norm1(x + dropout(attention_output))
        x = TODO = None

        # --- 5. FFN and Second Residual Connection ---
        ffn_output = self.ffn(x)
        # x = norm2(x + dropout(ffn_output))
        x = TODO = None

        return x


Transformer Component Simulation:
Input embeddings shape: torch.Size([2, 8, 64])

1. ATTENTION PROJECTIONS:
Query (Q) shape: torch.Size([2, 8, 64])
Key (K) shape: torch.Size([2, 8, 64])
Value (V) shape: torch.Size([2, 8, 64])
Attention scores shape: torch.Size([2, 8, 8])

2. FEED-FORWARD NETWORK:
Input to MLP: torch.Size([2, 8, 64])
After first linear layer: torch.Size([2, 8, 256])
After activation: torch.Size([2, 8, 256])
After second linear layer: torch.Size([2, 8, 64])

3. OUTPUT PROJECTION:
Final logits shape: torch.Size([2, 8, 10000])

All operations completed efficiently on AMD GPU!
Q proj: 4,096 parameters
K proj: 4,096 parameters
V proj: 4,096 parameters
FF layer 1: 16,640 parameters
FF layer 2: 16,448 parameters
Output proj: 650,000 parameters

Total parameters: 695,376
Memory usage: ~2.7 MB (float32)


In [ ]:
# Parameters
embed_dim = 64
num_heads = 8
ff_dim = embed_dim * 4
seq_length = 32
batch_size = 4

# Instantiate the block
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim).to(device)
print("--- Model Instantiated ---")
print(f"Running verification suite on device: {device}\n")


# --- Test 1: Shape Consistency ---
print("[Test 1/3] Running Shape Consistency Check...")
dummy_input = torch.randn(batch_size, seq_length, embed_dim, device=device)
output = transformer_block(dummy_input)
assert dummy_input.shape == output.shape, f"Shape mismatch! Input: {dummy_input.shape}, Output: {output.shape}"
print("✅ Passed: Output shape is consistent with input shape.")


# --- Test 2: Gradient Flow Check ---
print("\n[Test 2/3] Running Gradient Flow Check...")
transformer_block.zero_grad() # Ensure gradients are clean
output = transformer_block(dummy_input)
# Create a dummy loss and backpropagate
loss = output.mean()
loss.backward()
# Check that key parameters have gradients
grad_exists = transformer_block.qkv_proj.weight.grad is not None and transformer_block.ffn[0].weight.grad is not None
assert grad_exists, "Gradient flow failed! No gradients found in key layers."
print("✅ Passed: Gradients are flowing through the block.")


# --- Test 3: LayerNorm Output Statistics Check ---
print("\n[Test 3/3] Running LayerNorm Output Statistics Check...")
# The output of the block is the output of the second LayerNorm.
# Its mean should be close to 0 and its std dev should be close to 1 across the feature dimension.
output_mean = output.mean(dim=-1)
output_std = output.std(dim=-1)
mean_is_near_zero = torch.allclose(output_mean, torch.zeros_like(output_mean), atol=1e-5)
std_is_near_one = torch.allclose(output_std, torch.ones_like(output_std), atol=1e-1) # Std has more variance
assert mean_is_near_zero, f"LayerNorm failed! Mean of output is not close to 0."
assert std_is_near_one, f"LayerNorm failed! Standard deviation of output is not close to 1."
print("✅ Passed: Output statistics are consistent with Layer Normalization.")

print("\n\n🎉 All verification tests passed successfully!")

## Lab Summary

### Technical Concepts Learned
- **Linear Transformations**: The mathematical foundation y = xW^T + b
- **Parameter Management**: Weight initialization and device placement
- **Batch Processing**: Handling multiple samples simultaneously
- **Memory Efficiency**: Broadcasting vs explicit tensor expansion
- **Transformer Architecture**: How linear layers build complex models

### Experiment Further
Try these modifications to deepen your understanding:
- Change embedding dimensions and observe parameter scaling
- Experiment with different tensor shapes and broadcasting
- Compare CPU vs GPU computation times
- Implement custom linear layers from scratch